In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal, special
from ipywidgets import Checkbox, RadioButtons, IntSlider, VBox, HBox, Layout, HTML
from IPython.display import display

plt.ioff()

# ==============================================================================
# USAGE
#
# COMPARISON OF OTHER IDEAL-FILTER APPROXIMATIONS
#
# Approximations:
#
#       Transitional Butterworth-Bessel
#       Gaussian
#       Legendre
#       Hyperspherical
#
# Interactive parameter:
#
#       N = filter order, 1,...,10
#
# Displayed quantities:
#
#       Magnitude Response
#       Gain Function
#       Loss Function
#
# Fixed illustrative parameters:
#
#       ωp      = 1
#       ωc      = 1
#       ε       = 1
#       m_trans = 0.5
#       m_leg   = 0
#       α       = 0.5
#
# NOTE:
#
# Phase response and group delay are intentionally not included.
#
# The theory presented for Gaussian, Legendre, and hyperspherical filters
# specifies their power responses |H(jω)|², but does not uniquely specify
# the complete complex transfer function H(s).
#
# Magnitude alone is insufficient to determine phase and group delay unless
# an additional realization assumption (for example, minimum phase) is made.
# Such an assumption is outside the theory considered here.
#
# Papoulis and Halpern approximations are also omitted until their defining
# expressions are checked in detail.
# ==============================================================================

# ==============================================================================
# JUPYTER DISPLAY SETTINGS
# ==============================================================================

display(HTML("""
<style>

.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.output,
.output_area,
.output_subarea,
.output_scroll {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.jupyter-widgets,
.widget-box,
.widget-html,
.widget-html-content {
    overflow: visible !important;
    max-height: none !important;
}

.jp-Cell-outputWrapper {
    overflow: visible !important;
}

</style>
"""))

# ==============================================================================
# FIXED PARAMETERS
# ==============================================================================

wp = 1.0
wc = 1.0
epsilon = 1.0
transition_m = 0.5
legendre_m = 0
alpha = 0.5

omega = np.linspace(0.0, 2.5, 4000)

# ==============================================================================
# COLORS
# ==============================================================================

approximation_names = ['Transitional', 'Gaussian', 'Legendre', 'Hyperspherical']

colors = {
    'Transitional': 'red',
    'Gaussian': 'blue',
    'Legendre': 'green',
    'Hyperspherical': 'purple'
}

# ==============================================================================
# TRANSITIONAL FILTER
# ==============================================================================

def split_upper_real(poles):

    tol = 1e-10

    upper = sorted([p for p in poles if p.imag > tol], key=lambda z: z.imag, reverse=True)

    real = sorted([p for p in poles if abs(p.imag) <= tol], key=lambda z: z.real)

    return upper, real

def interpolate_pole(p1, p2, m):

    radius = np.exp((1.0 - m) * np.log(np.abs(p1)) + m * np.log(np.abs(p2)))

    angle = (1.0 - m) * np.angle(p1) + m * np.angle(p2)

    return radius * np.exp(1j * angle)

def transitional_magnitude(N, m, omega):

    z_butter, butter_poles, k_butter = signal.buttap(N)

    z_bessel, bessel_poles, k_bessel = signal.besselap(N, norm='phase')

    butter_upper, butter_real = split_upper_real(butter_poles)

    bessel_upper, bessel_real = split_upper_real(bessel_poles)

    upper = []

    for p1, p2 in zip(butter_upper, bessel_upper):

        upper.append(interpolate_pole(p1, p2, m))

    poles = []

    poles.extend(upper)

    for p1, p2 in zip(butter_real, bessel_real):

        poles.append(interpolate_pole(p1, p2, m))

    poles.extend([np.conjugate(p) for p in upper[::-1]])

    poles = np.asarray(poles, dtype=complex)

    denominator = np.poly(poles)

    denominator = np.real_if_close(denominator, tol=1000).real

    numerator = np.array([denominator[-1]])

    omega_response, H = signal.freqs(numerator, denominator, worN=omega)

    return np.abs(H)

# ==============================================================================
# GAUSSIAN MAGNITUDE
#
# Finite-order power-response approximation:
#
# |H(jω)|² =
#
#       1
# -------------------------
# 1 + Σ (γ^k/k!) ω^(2k)
#
# γ = ln(2)/ωc²
#
# Using the truncated Nth-order expression is essential here because the
# purpose of the notebook is to investigate the effect of filter order N.
# ==============================================================================

def gaussian_magnitude(N, omega):

    gamma = np.log(2.0) / wc**2

    denominator = np.ones_like(omega)

    for k in range(1, N + 1):

        denominator += (gamma**k / special.factorial(k)) * omega**(2 * k)

    power = 1.0 / denominator

    return np.sqrt(power)

# ==============================================================================
# LEGENDRE MAGNITUDE
#
# For the present illustrative comparison m = 0:
#
# |H(jω)|² =
#
#       1
# ------------------------
# 1 + ε²[P_N(ω/ωp)]²
# ==============================================================================

def legendre_magnitude(N, omega):

    x = omega / wp

    P = special.eval_legendre(N, x)

    power = 1.0 / (1.0 + epsilon**2 * P**2)

    return np.sqrt(power)

# ==============================================================================
# HYPERSPHERICAL MAGNITUDE
#
# F_N^(α)(x) =
#
# N! / [(1+α)(2+α)...(N+α)] P_N^(α,α)(x)
#
# |H(jω)|² =
#
#       1
# ------------------------
# 1 + ε²[F_N^(α)(x)]²
# ==============================================================================

def hyperspherical_magnitude(N, omega):

    x = omega / wp

    normalization = special.factorial(N) * special.gamma(1.0 + alpha) / special.gamma(N + 1.0 + alpha)

    jacobi = special.eval_jacobi(N, alpha, alpha, x)

    F = normalization * jacobi

    power = 1.0 / (1.0 + epsilon**2 * F**2)

    return np.sqrt(power)

# ==============================================================================
# CALCULATE ALL MAGNITUDE RESPONSES
# ==============================================================================

def calculate_responses(N):

    new_responses = {}

    new_responses['Transitional'] = transitional_magnitude(N, transition_m, omega)

    new_responses['Gaussian'] = gaussian_magnitude(N, omega)

    new_responses['Legendre'] = legendre_magnitude(N, omega)

    new_responses['Hyperspherical'] = hyperspherical_magnitude(N, omega)

    return new_responses

# ==============================================================================
# INITIAL ORDER
# ==============================================================================

N_initial = 4

responses = calculate_responses(N_initial)

# ==============================================================================
# CHECKBOXES
# ==============================================================================

cb_trans = Checkbox(value=True, description='Transitional', indent=False, layout=Layout(width='145px'))

cb_gauss = Checkbox(value=True, description='Gaussian', indent=False, layout=Layout(width='145px'))

cb_legendre = Checkbox(value=True, description='Legendre', indent=False, layout=Layout(width='145px'))

cb_hyper = Checkbox(value=True, description='Hyperspherical', indent=False, layout=Layout(width='145px'))

checkboxes = {
    'Transitional': cb_trans,
    'Gaussian': cb_gauss,
    'Legendre': cb_legendre,
    'Hyperspherical': cb_hyper
}

# ==============================================================================
# COLOR INDICATORS
# ==============================================================================

def color_indicator(color):

    return HTML(f"""
    <div style="
        width:40px;
        height:12px;
        display:flex;
        align-items:center;
        margin-left:4px;
    ">
        <div style="
            width:32px;
            height:3px;
            background:{color};
            border-radius:2px;
        "></div>
    </div>
    """, layout=Layout(width='45px'))

row_trans = HBox([cb_trans, color_indicator(colors['Transitional'])], layout=Layout(align_items='center'))

row_gauss = HBox([cb_gauss, color_indicator(colors['Gaussian'])], layout=Layout(align_items='center'))

row_legendre = HBox([cb_legendre, color_indicator(colors['Legendre'])], layout=Layout(align_items='center'))

row_hyper = HBox([cb_hyper, color_indicator(colors['Hyperspherical'])], layout=Layout(align_items='center'))

# ==============================================================================
# CURRENT PARAMETERS PANEL
# ==============================================================================

parameter_html = HTML()

def update_parameter_html(N):

    parameter_html.value = f"""
    <div style="
        margin-top:8px;
        padding-top:8px;
        border-top:1px solid #dddddd;
        font-size:11px;
        line-height:1.65;
    ">

    <b>Current parameters</b><br>

    N =
    <span style="color:#0066cc;"><b>{N}</b></span><br>

    ω<sub>p</sub> = {wp:.1f}<br>

    ω<sub>c</sub> = {wc:.1f}<br>

    ε = {epsilon:.1f}<br>

    Transitional m = {transition_m:.1f}<br>

    Legendre m = {legendre_m}<br>

    Hyperspherical α = {alpha:.1f}

    </div>
    """

update_parameter_html(N_initial)

# ==============================================================================
# APPROXIMATION-SELECTION PANEL
# ==============================================================================

control_title = HTML("""
<div style="
    font-size:13px;
    font-weight:bold;
    margin-bottom:7px;
">
Approximation Selection
</div>
""")

control_panel = VBox(
    [
        control_title,
        row_trans,
        row_gauss,
        row_legendre,
        row_hyper,
        parameter_html
    ],
    layout=Layout(
        width='245px',
        min_width='245px',
        max_width='245px',
        border='1px solid #cccccc',
        padding='10px',
        align_items='flex-start'
    )
)

# ==============================================================================
# DISPLAY SELECTOR
# ==============================================================================

display_title = HTML("""
<div style="
    font-size:13px;
    font-weight:bold;
    margin-bottom:7px;
">
Displayed Quantity
</div>
""")

display_selector = RadioButtons(
    options=[
        'Magnitude Response',
        'Gain Function',
        'Loss Function'
    ],
    value='Magnitude Response',
    description='',
    layout=Layout(width='165px')
)

# ==============================================================================
# DISPLAY PANEL
#
# The left margin moves the panel slightly to the right so that it does not
# touch the boundary of the graph.
# ==============================================================================

display_panel = VBox(
    [
        display_title,
        display_selector
    ],
    layout=Layout(
        width='190px',
        min_width='190px',
        max_width='190px',
        border='1px solid #cccccc',
        padding='10px',
        margin='0px 0px 0px 18px',
        align_items='flex-start'
    )
)

# ==============================================================================
# ORDER SLIDER
# ==============================================================================

order_slider = IntSlider(
    value=N_initial,
    min=1,
    max=10,
    step=1,
    description='Filter Order N:',
    continuous_update=True,
    readout=True,
    style={'description_width':'95px'},
    layout=Layout(width='650px')
)

# ==============================================================================
# DESCRIPTION
# ==============================================================================

description = HTML("""
<div style="
    border:1px solid #9ec9f5;
    border-radius:7px;
    padding:9px 11px;
    margin-bottom:8px;
    font-size:12px;
    line-height:1.5;
    background:#f7fbff;
    width:1320px;
    max-width:1320px;
    box-sizing:border-box;
">

<b>Comparison of Other Ideal-Filter Approximations</b><br>

This notebook compares the magnitude-based characteristics of transitional,
Gaussian, Legendre and hyperspherical approximations.

The filter order can be varied from <b>N = 1</b> to <b>N = 10</b>.

<br>

<b>Interpretation:</b>
Use the checkboxes to select the approximations, the radio buttons to select
magnitude response, gain function or loss function, and the horizontal slider
to investigate the effect of filter order.

Phase response and group delay are intentionally not included because the
power-response equations given for Gaussian, Legendre and hyperspherical
filters do not uniquely determine the complete complex transfer function.

</div>
""", layout=Layout(width='1330px', max_width='1330px'))

# ==============================================================================
# CREATE FIGURE ONCE
# ==============================================================================

fig, ax = plt.subplots(figsize=(8.8, 5.4))

lines = {}

for name in approximation_names:

    lines[name], = ax.plot([], [], linewidth=2.2, color=colors[name], label=name)

normalized_frequency_line = ax.axvline(1.0, color='black', linestyle=':', linewidth=1.0)

minus3db_line = ax.axhline(1.0 / np.sqrt(2.0), color='gray', linestyle='--', linewidth=0.9)

ax.grid(True, linestyle=':', alpha=0.5)

ax.tick_params(axis='both', labelsize=9)

fig.subplots_adjust(left=0.11, right=0.96, bottom=0.15, top=0.88)

fig.canvas.header_visible = False

fig.canvas.toolbar_visible = False

fig.canvas.resizable = False

fig.canvas.layout.width = '850px'

fig.canvas.layout.height = '530px'

# ==============================================================================
# EMPTY-SELECTION MESSAGE
# ==============================================================================

empty_message = ax.text(
    0.5,
    0.5,
    'Select at least one approximation.',
    transform=ax.transAxes,
    horizontalalignment='center',
    verticalalignment='center',
    fontsize=13,
    fontweight='bold',
    bbox=dict(boxstyle='round,pad=0.7', facecolor='white', edgecolor='gray', alpha=0.95)
)

empty_message.set_visible(False)

# ==============================================================================
# UPDATE PLOT
# ==============================================================================

def update_plot(change=None):

    selected_quantity = display_selector.value

    active_count = 0

    for name in approximation_names:

        magnitude = responses[name]

        if selected_quantity == 'Magnitude Response':

            y_values = magnitude

        elif selected_quantity == 'Gain Function':

            y_values = 20.0 * np.log10(np.maximum(magnitude, 1e-15))

        elif selected_quantity == 'Loss Function':

            y_values = -20.0 * np.log10(np.maximum(magnitude, 1e-15))

        if checkboxes[name].value:

            lines[name].set_data(omega, y_values)

            active_count += 1

        else:

            lines[name].set_data([], [])

    # --------------------------------------------------------------------------
    # EMPTY SELECTION
    # --------------------------------------------------------------------------

    if active_count == 0:

        empty_message.set_visible(True)

        display_selector.disabled = True

        normalized_frequency_line.set_visible(False)

        minus3db_line.set_visible(False)

        ax.set_xlim(0.0, 1.0)

        ax.set_ylim(0.0, 1.0)

        ax.set_xlabel('')

        ax.set_ylabel('')

        ax.set_title('Ideal-Filter Approximation Comparison', fontsize=13, fontweight='bold', pad=8)

        ax.grid(False)

        fig.canvas.draw_idle()

        return

    # --------------------------------------------------------------------------
    # ACTIVE APPROXIMATIONS
    # --------------------------------------------------------------------------

    empty_message.set_visible(False)

    display_selector.disabled = False

    normalized_frequency_line.set_visible(True)

    ax.set_xlim(0.0, 2.5)

    # --------------------------------------------------------------------------
    # MAGNITUDE
    # --------------------------------------------------------------------------

    if selected_quantity == 'Magnitude Response':

        minus3db_line.set_visible(True)

        minus3db_line.set_ydata([1.0 / np.sqrt(2.0), 1.0 / np.sqrt(2.0)])

        ax.set_ylim(0.0, 1.08)

        ax.set_xlabel('Normalized Angular Frequency ω/ωp', fontsize=10)

        ax.set_ylabel('|H(jω)|', fontsize=10)

        ax.set_title('Magnitude Comparison of Other Ideal-Filter Approximations', fontsize=13, fontweight='bold', pad=8)

    # --------------------------------------------------------------------------
    # GAIN
    # --------------------------------------------------------------------------

    elif selected_quantity == 'Gain Function':

        minus3db_line.set_visible(True)

        minus3db_line.set_ydata([-3.0103, -3.0103])

        ax.set_ylim(-80.0, 5.0)

        ax.set_xlabel('Normalized Angular Frequency ω/ωp', fontsize=10)

        ax.set_ylabel('Gain G(ω) (dB)', fontsize=10)

        ax.set_title('Gain-Function Comparison of Other Ideal-Filter Approximations', fontsize=13, fontweight='bold', pad=8)

    # --------------------------------------------------------------------------
    # LOSS
    # --------------------------------------------------------------------------

    elif selected_quantity == 'Loss Function':

        minus3db_line.set_visible(True)

        minus3db_line.set_ydata([3.0103, 3.0103])

        ax.set_ylim(0.0, 80.0)

        ax.set_xlabel('Normalized Angular Frequency ω/ωp', fontsize=10)

        ax.set_ylabel('Loss A(ω) (dB)', fontsize=10)

        ax.set_title('Loss-Function Comparison of Other Ideal-Filter Approximations', fontsize=13, fontweight='bold', pad=8)

    ax.grid(True, linestyle=':', alpha=0.5)

    fig.canvas.draw_idle()

# ==============================================================================
# UPDATE FILTER ORDER
# ==============================================================================

def update_order(change=None):

    global responses

    N = order_slider.value

    responses = calculate_responses(N)

    update_parameter_html(N)

    update_plot()

# ==============================================================================
# CALLBACKS
# ==============================================================================

cb_trans.observe(update_plot, names='value')

cb_gauss.observe(update_plot, names='value')

cb_legendre.observe(update_plot, names='value')

cb_hyper.observe(update_plot, names='value')

display_selector.observe(update_plot, names='value')

order_slider.observe(update_order, names='value')

# ==============================================================================
# INITIALIZE
# ==============================================================================

update_plot()

# ==============================================================================
# SLIDER PANEL
# ==============================================================================

order_slider_box = HBox(
    [order_slider],
    layout=Layout(
        width='850px',
        padding='0px 0px 0px 75px',
        box_sizing='border-box',
        justify_content='flex-start',
        align_items='center',
        margin='-7px 0px 0px 0px'
    )
)

# ==============================================================================
# PLOT COLUMN
# ==============================================================================

plot_column = VBox(
    [
        fig.canvas,
        order_slider_box
    ],
    layout=Layout(
        width='850px',
        min_width='850px',
        max_width='850px',
        align_items='center',
        justify_content='flex-start'
    )
)

# ==============================================================================
# MAIN LAYOUT
# ==============================================================================

main_layout = HBox(
    [
        control_panel,
        plot_column,
        display_panel
    ],
    layout=Layout(
        width='1320px',
        align_items='flex-start',
        justify_content='flex-start'
    )
)

# ==============================================================================
# DISPLAY
# ==============================================================================

display(description)

display(main_layout)